# Sketch to Image Inference

In [ ]:
import os
import glob
from pathlib import Path
from PIL import Image, ImageOps
import numpy as np
import torch
import torch.nn as nn
from torch.nn import init
import functools
import shutil
import zipfile
import matplotlib.pyplot as plt
from torchvision.utils import save_image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Model Architecture Components

In [ ]:
class Identity(nn.Module):
    def forward(self, x):
        return x

def get_norm_layer(norm_type="instance"):
    if norm_type == "batch":
        norm_layer = functools.partial(nn.BatchNorm2d, affine=True, track_running_stats=True)
    elif norm_type == "syncbatch":
        norm_layer = functools.partial(nn.SyncBatchNorm, affine=True, track_running_stats=True)
    elif norm_type == "instance":
        norm_layer = functools.partial(nn.InstanceNorm2d, affine=False, track_running_stats=False)
    elif norm_type == "none":
        def norm_layer(x):
            return Identity()
    else:
        raise NotImplementedError("normalization layer [%s] is not found" % norm_type)
    return norm_layer

def init_weights(net, init_type="normal", init_gain=0.02):
    def init_func(m):
        classname = m.__class__.__name__
        if hasattr(m, "weight") and (classname.find("Conv") != -1 or classname.find("Linear") != -1):
            if init_type == "normal":
                init.normal_(m.weight.data, 0.0, init_gain)
            elif init_type == "xavier":
                init.xavier_normal_(m.weight.data, gain=init_gain)
            elif init_type == "kaiming":
                init.kaiming_normal_(m.weight.data, a=0, mode="fan_in")
            elif init_type == "orthogonal":
                init.orthogonal_(m.weight.data, gain=init_gain)
            else:
                raise NotImplementedError("initialization method [%s] is not implemented" % init_type)
            if hasattr(m, "bias") and m.bias is not None:
                init.constant_(m.bias.data, 0.0)
        elif classname.find("BatchNorm2d") != -1:
            init.normal_(m.weight.data, 1.0, init_gain)
            init.constant_(m.bias.data, 0.0)

    print("initialize network with %s" % init_type)
    net.apply(init_func)

def define_G(input_nc, output_nc, ngf, netG, norm="batch", use_dropout=False, init_type="normal", init_gain=0.02):
    net = None
    norm_layer = get_norm_layer(norm_type=norm)

    if netG == "unet_128":
        net = UnetGenerator(input_nc, output_nc, 7, ngf, norm_layer=norm_layer, use_dropout=use_dropout)
    elif netG == "unet_256":
        net = UnetGenerator(input_nc, output_nc, 8, ngf, norm_layer=norm_layer, use_dropout=use_dropout)
    else:
        raise NotImplementedError("Generator model name [%s] is not recognized" % netG)
    
    init_weights(net, init_type, init_gain=init_gain)
    return net

class UnetGenerator(nn.Module):
    def __init__(self, input_nc, output_nc, num_downs, ngf=64, norm_layer=nn.BatchNorm2d, use_dropout=False):
        super(UnetGenerator, self).__init__()
        unet_block = UnetSkipConnectionBlock(ngf * 8, ngf * 8, input_nc=None, submodule=None, norm_layer=norm_layer, innermost=True)
        for i in range(num_downs - 5):
            unet_block = UnetSkipConnectionBlock(ngf * 8, ngf * 8, input_nc=None, submodule=unet_block, norm_layer=norm_layer, use_dropout=use_dropout)
        unet_block = UnetSkipConnectionBlock(ngf * 4, ngf * 8, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        unet_block = UnetSkipConnectionBlock(ngf * 2, ngf * 4, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        unet_block = UnetSkipConnectionBlock(ngf, ngf * 2, input_nc=None, submodule=unet_block, norm_layer=norm_layer)
        self.model = UnetSkipConnectionBlock(output_nc, ngf, input_nc=input_nc, submodule=unet_block, outermost=True, norm_layer=norm_layer)

    def forward(self, input):
        return self.model(input)

class UnetSkipConnectionBlock(nn.Module):
    def __init__(self, outer_nc, inner_nc, input_nc=None, submodule=None, outermost=False, innermost=False, norm_layer=nn.BatchNorm2d, use_dropout=False):
        super(UnetSkipConnectionBlock, self).__init__()
        self.outermost = outermost
        if type(norm_layer) == functools.partial:
            use_bias = norm_layer.func == nn.InstanceNorm2d
        else:
            use_bias = norm_layer == nn.InstanceNorm2d
        if input_nc is None:
            input_nc = outer_nc
        downconv = nn.Conv2d(input_nc, inner_nc, kernel_size=4, stride=2, padding=1, bias=use_bias)
        downrelu = nn.LeakyReLU(0.2, True)
        downnorm = norm_layer(inner_nc)
        uprelu = nn.ReLU(True)
        upnorm = norm_layer(outer_nc)

        if outermost:
            upconv = nn.ConvTranspose2d(inner_nc * 2, outer_nc, kernel_size=4, stride=2, padding=1)
            down = [downconv]
            up = [uprelu, upconv, nn.Tanh()]
            model = down + [submodule] + up
        elif innermost:
            upconv = nn.ConvTranspose2d(inner_nc, outer_nc, kernel_size=4, stride=2, padding=1, bias=use_bias)
            down = [downrelu, downconv]
            up = [uprelu, upconv, upnorm]
            model = down + up
        else:
            upconv = nn.ConvTranspose2d(inner_nc * 2, outer_nc, kernel_size=4, stride=2, padding=1, bias=use_bias)
            down = [downrelu, downconv, downnorm]
            up = [uprelu, upconv, upnorm]

            if use_dropout:
                model = down + [submodule] + up + [nn.Dropout(0.5)]
            else:
                model = down + [submodule] + up

        self.model = nn.Sequential(*model)

    def forward(self, x):
        if self.outermost:
            return self.model(x)
        else:
            return torch.cat([x, self.model(x)], 1)

## Inference Functions

In [ ]:
def load_generator_weights(ckpt_path, generator, strict=True):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {ckpt_path}')
    print(f"Loading weights from {ckpt_path}...")
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    if 'generator_state_dict' in ckpt:
        generator.load_state_dict(ckpt['generator_state_dict'], strict=strict)
    elif 'netG_state_dict' in ckpt:
        generator.load_state_dict(ckpt['netG_state_dict'], strict=strict)
    else:
        generator.load_state_dict(ckpt, strict=strict)
    print("Weights loaded successfully.")

def smart_pad_and_resize(img: Image.Image, target_size: int = 256) -> Image.Image:
    w, h = img.size
    max_side = max(w, h)
    canvas_side = max(max_side, target_size)
    pad_left = (canvas_side - w) // 2
    pad_top = (canvas_side - h) // 2
    pad_right = canvas_side - w - pad_left
    pad_bottom = canvas_side - h - pad_top
    img_padded = ImageOps.expand(
        img,
        border=(pad_left, pad_top, pad_right, pad_bottom),
        fill=(255, 255, 255),
    )
    if img_padded.size != (target_size, target_size):
        img_padded = img_padded.resize((target_size, target_size), Image.BICUBIC)
    return img_padded

def preprocess_image(image_path, target_size=256):
    with Image.open(image_path) as src_img:
        img = src_img.convert("RGB")
    img = smart_pad_and_resize(img, target_size=target_size)
    arr = np.asarray(img, dtype=np.float32) / 127.5 - 1.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return tensor

def denormalize_tensor(x):
    return (x.detach().cpu().clamp(-1, 1) + 1.0) / 2.0

## Run Inference

In [ ]:
# Configuration
CHECKPOINT_PATH = '/kaggle/input/pix2pix-weights/latest_net_G.pth' # Change this to your weights path
INPUT_PATH = '/kaggle/input/sketch-images' # Can be a single image path or a directory path
OUTPUT_DIR = '/kaggle/working/generated_images'
ZIP_OUTPUT_PATH = '/kaggle/working/generated_images.zip'

# 1. Initialize and load model
generator = define_G(input_nc=3, output_nc=3, ngf=64, netG='unet_256', norm='batch', use_dropout=False, init_type='normal', init_gain=0.02).to(DEVICE)
generator.eval()

try:
    load_generator_weights(CHECKPOINT_PATH, generator)
except Exception as e:
    print(f"Failed to load weights: {e}")

# 2. Gather input images
if os.path.isfile(INPUT_PATH):
    image_paths = [INPUT_PATH]
elif os.path.isdir(INPUT_PATH):
    image_paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp']:
        image_paths.extend(glob.glob(os.path.join(INPUT_PATH, ext)))
        image_paths.extend(glob.glob(os.path.join(INPUT_PATH, ext.upper())))
else:
    image_paths = []
    print("Invalid INPUT_PATH")

print(f"Found {len(image_paths)} images to process.")

# 3. Process images
os.makedirs(OUTPUT_DIR, exist_ok=True)
with torch.no_grad():
    for img_path in image_paths:
        try:
            tensor = preprocess_image(img_path).to(DEVICE)
            output = generator(tensor)
            output_denorm = denormalize_tensor(output[0])
            
            filename = os.path.basename(img_path)
            out_path = os.path.join(OUTPUT_DIR, filename)
            save_image(output_denorm, out_path)
            print(f"Generated and saved: {filename}")
        except Exception as e:
            print(f"Error processing {img_path}: {e}")

# 4. Zip the output directory
if os.path.exists(OUTPUT_DIR) and len(os.listdir(OUTPUT_DIR)) > 0:
    print(f"Zipping outputs to {ZIP_OUTPUT_PATH}...")
    with zipfile.ZipFile(ZIP_OUTPUT_PATH, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(OUTPUT_DIR):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, OUTPUT_DIR)
                zipf.write(file_path, arcname)
    print("Done! You can now download the zip file.")
else:
    print("No images were generated.")


In [ ]:
# 5. Show sketch and generated images side-by-side
import glob
import matplotlib.pyplot as plt
import os
from PIL import Image

generated_images = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*')))
if len(generated_images) > 0:
    num_show = min(4, len(generated_images)) # Display up to 4 pairs
    fig, axes = plt.subplots(num_show, 2, figsize=(8, 4 * num_show))
    if num_show == 1:
        axes = [axes]
    for i in range(num_show):
        gen_path = generated_images[i]
        filename = os.path.basename(gen_path)
        
        # Try to find corresponding sketch path in INPUT_PATH
        sketch_path = os.path.join(INPUT_PATH, filename) if os.path.isdir(INPUT_PATH) else INPUT_PATH
        
        # Show Sketch
        if os.path.exists(sketch_path) and os.path.isfile(sketch_path):
            sketch_img = Image.open(sketch_path)
            axes[i][0].imshow(sketch_img)
            axes[i][0].set_title("Sketch")
        else:
            axes[i][0].text(0.5, 0.5, "Sketch not found", ha='center', va='center')
        axes[i][0].axis('off')
        
        # Show Generated
        gen_img = Image.open(gen_path)
        axes[i][1].imshow(gen_img)
        axes[i][1].set_title("Generated: " + filename)
        axes[i][1].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No generated images found in", OUTPUT_DIR)